# RAG-based Machine Learning Knowledge Assistant:

**Objective:**  
Build an end-to-end Retrieval-Augmented Generation (RAG) system that answers Machine Learning questions using *only* the content from the book **Introduction to Machine Learning with Python**.

**Knowledge Source:**  
- intro-to-ml.pdf

**Key Capabilities:**
- PDF ingestion and chunking
- Vector embedding and similarity search
- Grounded answer generation using retrieved context

## Environment Setup

This section installs all required libraries for building a
Retrieval-Augmented Generation (RAG) system, including:
- PDF loading
- Text chunking
- Embedding generation
- Vector similarity search
- LLM-based answer generation

In [21]:
# Core libraries
# Install all dependencies required for the RAG pipeline.
# - langchain & langchain_community & langchain_huggingface: orchestration framework and specific embedding implementations
# - pypdf: PDF text extraction
# - sentence-transformers: embedding models
# - faiss-cpu: vector similarity search
# - transformers & torch: LLM inference

# Install FFmpeg and related libraries required by torchcodec (an indirect dependency of sentence-transformers)
!apt-get update && apt-get install -y ffmpeg libsm6 libxext6

# Upgrade torch and transformers to ensure compatibility with model weights
!pip install --upgrade torch transformers

# Reinstall other dependencies to ensure no conflicts after torch/transformers upgrade
!pip install langchain langchain_community langchain_huggingface pypdf sentence-transformers faiss-cpu

# Uninstall torchcodec as it's causing incompatibility issues with the current PyTorch version
!pip uninstall -y torchcodec

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://cli.github.com/packages stable InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,991 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Ign:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:11 h

## Step 1: PDF Loading and Text Extraction

In this step, the Machine Learning book (intro-to-ml.pdf) is loaded
and converted into page-wise text documents.

In [22]:
# Import the PDF loader utility from LangChain Community
# The previous PyPDFLoader was causing issues with torchcodec and ffmpeg dependencies.
# We will now use pypdf directly to load the document content.
import pypdf
from langchain_core.documents import Document

# Load the machine learning book PDF using pypdf
reader = pypdf.PdfReader("/content/intro-to-ml.pdf")

# Extract text page-by-page and create LangChain Document objects
documents = []
for i, page in enumerate(reader.pages):
    page_content = page.extract_text()
    if page_content:
        documents.append(Document(page_content=page_content, metadata={"source": "/content/intro-to-ml.pdf", "page": i}))

# Display basic information for validation
print(f"Total pages loaded: {len(documents)}")
print(documents[0].page_content[:1000])

Total pages loaded: 387
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS



## Step 2: Text Chunking

The extracted text is split into overlapping chunks to enable
semantic retrieval while preserving contextual continuity.

In [23]:
# Import text splitter for recursive chunking
# Importing directly from langchain_text_splitters.character to avoid indirect sentence-transformers/torchcodec dependency issues
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# Define chunking strategy:
# - chunk_size: number of characters per chunk
# - chunk_overlap: overlapping characters between chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

# Split full document into smaller chunks
chunks = text_splitter.split_documents(documents)

# Verify chunking results
print(f"Total chunks created: {len(chunks)}")
print("Sample chunk content:\n")
print(chunks[0].page_content[:500])

Total chunks created: 1143
Sample chunk content:

Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


## Step 3: Text Embedding Generation

Each text chunk is converted into a dense vector representation
using a pretrained

In [24]:
# Import HuggingFace embedding wrapper from LangChain
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Initialize embedding model
# all-MiniLM-L6-v2 is lightweight and well-suited for semantic search
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Step 4: Vector Store Creation

The generated text embeddings are stored in a FAISS (Facebook AI Similarity Search) vector store, which enables fast similarity lookups to retrieve relevant context for user queries.

FAISS is used to store embeddings and perform fast similarity search over the chunked document corpus

In [25]:
# Import FAISS vector store
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store from the generated text chunks and embedding model
vector_store = FAISS.from_documents(chunks, embedding_model)

print("FAISS vector store created successfully.")
print(f"Number of vectors in the store: {vector_store.index.ntotal}")

FAISS vector store created successfully.
Number of vectors in the store: 1143


## Step 5: Semantic Retrieval

User queries are converted into embeddings and used to retrieve
the most relevant text chunks from the vector database.

In [26]:
# Define a sample user query
query = "What is overfitting in machine learning?"

# Perform similarity search to retrieve top-k relevant chunks
# Reducing k from 5 to 3 to provide less context to the LLM
retrieved_docs = vector_store.similarity_search(query, k=3)

# Display retrieved context for inspection
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content[:400])


--- Retrieved Chunk 1 ---
occurs when you fit a model too closely to the particularities of the training set and
obtain a model that works well on the training set but is not able to generalize to new
data. On the other hand, if your model is too simple—say, “Everybody who owns a
house buys a boat”—then you might not be able to capture all the aspects of and vari‐
ability in the data, and your model will do badly even on t

--- Retrieved Chunk 2 ---
much on each individual data point in our training set, and the model will not gener‐
alize well to new data.
There is a sweet spot in between that will yield the best generalization performance.
This is the model we want to find.
The trade-off between overfitting and underfitting is illustrated in Figure 2-1.
28 | Chapter 2: Supervised Learning

--- Retrieved Chunk 3 ---
Supervised Machine Learning Algorithms
We will now review the most popular machine learning algorithms and explain how
they learn from data and how they make predictions.

In [27]:
# Display the full content of the retrieved documents for inspection
print("Inspecting Retrieved Documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Full Retrieved Chunk {i} ---")
    print(doc.page_content)

Inspecting Retrieved Documents:

--- Full Retrieved Chunk 1 ---
occurs when you fit a model too closely to the particularities of the training set and
obtain a model that works well on the training set but is not able to generalize to new
data. On the other hand, if your model is too simple—say, “Everybody who owns a
house buys a boat”—then you might not be able to capture all the aspects of and vari‐
ability in the data, and your model will do badly even on the training set. Choosing
too simple a model is called underfitting.
The more complex we allow our model to be, the better we will be able to predict on
the training data. However, if our model becomes too complex, we start focusing too
much on each individual data point in our training set, and the model will not gener‐
alize well to new data.

--- Full Retrieved Chunk 2 ---
much on each individual data point in our training set, and the model will not gener‐
alize well to new data.
There is a sweet spot in between that will yiel

## Step 6: Retrieval-Augmented Answer Generation

The retrieved chunks are injected into a prompt that instructs
the LLM to answer strictly based on the provided context

In [28]:
# Import necessary transformers components
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Model details
model_id = "google/flan-t5-large"

# Explicitly load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Removed tie_word_embeddings=False to allow default loading behavior, potentially fixing MISSING weights issue
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Define a grounded prompt template
prompt_template = """
Context:
{context}

Question:
{question}

As a Machine Learning expert, using ONLY the information contained in the Context, give a comprehensive answer to the question.
Respond only to the question asked, response should be consise and relevant to the question.
Answer:
"""

# Combine retrieved text into a single context block
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Format final prompt
final_prompt = prompt_template.format(
    context=context_text,
    question=query
)

# Prepare input for the model
inputs = tokenizer(final_prompt, return_tensors="pt", truncation=True, max_length=tokenizer.model_max_length)

# Determine device and move model/inputs to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate answer using model.generate()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512, # Controls the maximum number of tokens to generate
        num_beams=4,         # Use beam search for higher quality output
        no_repeat_ngram_size=3 # Prevent repeating n-grams to improve answer quality
        # Removed early_stopping=True to encourage more comprehensive answers
    )

# Decode the generated tokens to get the response string
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print the generated answer
print(f"Generated Answer: {response}")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated Answer: The more complex we allow our model to be, the better we will be able to predict on the training data. However, if our model becomes too complex, we start focusing too much on each individual data point in our training set, and the model will not gener alize well to new data.
